# 🏨 Hotel Booking Predictive Analytics

En este cuaderno se realiza la primera fase de exploración y auditoría diagnóstica sobre el dataset original de reservas hoteleras. El objetivo fundamental es comprender la estructura geométrica de la información, detectar inconsistencias operativas y validar la viabilidad estadística de los datos antes de alimentar el pipeline de Machine Learning.

## ⚙️ 1. Environment Setup & Data Ingestion

### Dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Seaborn configuration
sns.set_theme(style="whitegrid")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout


### Load the dataset

In [ ]:
path_data = "../data/raw/hotel_bookings.csv"
df_hotel = pd.read_csv(path_data)


### Global Display Configuration

In [ ]:
# Pandas display configuration: prevent truncation of wide dataframes
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 📈 2. Exploratory Data Analysis (EDA)

### 2.1 Structural Profiling & Dimensionality

In [ ]:
# Display the first 5 rows and dataset dimensions
print(f"The dataset has {df_hotel.shape[0]} rows and {df_hotel.shape[1]} columns.")
df_hotel.head()

In [ ]:
# Display dataframe summary: columns, data types, and non-null counts
df_hotel.info()

### 2.2 Feature Type Distribution

In [ ]:
# ----------- Data Type -----------
# Count of features grouped by data type
df_hotel.dtypes.value_counts()

In [ ]:
# -------------- Categorical Features -----------
# List of categorical/text columns
categorical_cols = df_hotel.select_dtypes(include=["object", "str"]).columns.tolist()

categorical_cols

In [ ]:
# -------------- Numerical Features --------------
# List of numerical columns
numerical_cols = df_hotel.select_dtypes(include=["int64", "float64"]).columns.tolist()

numerical_cols

### 2.3 Cardinality Audit

In [ ]:
# -------------- Cardinality of Categorical Features --------------
# Number of unique values per categorical feature
df_hotel[categorical_cols].nunique().sort_values(ascending=False)

### 2.4 Statistical Summaries & Anomaly Detection

In [ ]:
# 1. Statistical summary for numerical features
print("--- Numerical Features Statistical Summary ---")
print(df_hotel.describe().T)

In [ ]:
# 2. Statistical summary for categorical features
print("\n--- Categorical Features Statistical Summary ---")
print(df_hotel.describe(include=['object','str']).T)

### 2.5 Data Integrity & Missing Values

In [ ]:
# Filter and display only columns with missing values
nulls_summary = pd.DataFrame({
    "missing_values": df_hotel.isnull().sum(),
    "null_percentage": df_hotel.isnull().mean() * 100,
    "data_type": df_hotel.dtypes
})

# Filter and show only columns with missing values
nulls_summary[nulls_summary["missing_values"] > 0].sort_values("null_percentage", ascending=False)

### 2.6 Potential Data Leakage Audit

Las variables `reservation_status` y `reservation_status_date` se excluirán explícitamente del proceso de entrenamiento de los modelos debido a las siguientes restricciones críticas:

* **`reservation_status`:** Contiene el estado operativo final de la reserva (por ejemplo, `Check-Out`, `Canceled` o `No-Show`). Esta característica está directa y perfectamente correlacionada con nuestra variable objetivo `is_canceled`, lo que representa un caso clásico de redundancia de *target*.
* **`reservation_status_date`:** Registra la fecha exacta en la que se actualizó el estado final, por lo que también puede incluir información posterior al momento real de predicción.


In [ ]:
# Audit the distribution of reservation_status to verify target redundancy (Data Leakage check)
df_hotel['reservation_status'].value_counts(dropna=False)

In [ ]:
# Drop columns that cause Data Leakage
columns_to_drop = ['reservation_status', 'reservation_status_date']
df_hotel = df_hotel.drop(columns=columns_to_drop)

print(f"Columns dropped successfully. Current column count: {df_hotel.shape[1]}")

## 📊 3. Bivariate Analysis & Target Feature Correlations

### 3.1 Target Class Balance

In [ ]:
# Count the occurrences of each class (0 = Not Canceled, 1 = Canceled)
cancellation_counts = df_hotel['is_canceled'].value_counts()

# Print the results
print("Cancellation Distribution:")
print(cancellation_counts)

# Get the percentage 
cancellation_percentages = df_hotel['is_canceled'].value_counts(normalize=True) * 100
print("\nCancellation Percentages:")
print(cancellation_percentages)

# Visualization
plt.figure(figsize=(6, 4))
ax = sns.countplot(
    data=df_hotel, x='is_canceled', 
    hue='is_canceled', 
    palette=["lightblue", "lightgreen"], 
    legend=False
    )

plt.xticks(ticks=[0, 1], labels=["Not Canceled (0)", "Canceled (1)"])

plt.title("Distribution of the Target Variable: is_canceled")
plt.xlabel("Booking Canceled")
plt.ylabel("Number of Records")

plt.show()

### 3.2 Cancellation Drivers by Feature

In [ ]:
# ----- Hotel Type vs. Cancellation Rate -----
# Calculate cancellation percentage by hotel type
cancel_by_hotel = df_hotel.groupby("hotel")["is_canceled"].mean().sort_values(ascending=False) * 100
print("--- Cancellation Rate by Hotel Type ---")
print(cancel_by_hotel)

# Plot: Cancellation rate by hotel type
plt.figure(figsize=(6, 4))
cancel_by_hotel.plot(kind="bar")
plt.title("Cancellation Percentage by Hotel Type")
plt.xlabel("Hotel Type")
plt.ylabel("Cancellations (%)")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# ----- Deposit Type vs. Cancellation Rate -----
# Calculate cancellation percentage by deposit type
cancel_by_deposit = df_hotel.groupby("deposit_type")["is_canceled"].mean().sort_values(ascending=False) * 100
print("\n--- Cancellation Rate by Deposit Type ---")
print(cancel_by_deposit)

# Plot: Cancellation rate by deposit type
plt.figure(figsize=(6, 4))
cancel_by_deposit.plot(kind="bar")
plt.title("Cancellation Percentage by Deposit Type")
plt.xlabel("Deposit Type")
plt.ylabel("Cancellations (%)")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# ----- Market Segment vs. Cancellation Rate -----
# Calculate cancellation percentage by market segment
cancel_by_market = df_hotel.groupby("market_segment")["is_canceled"].mean().sort_values(ascending=False) * 100
print("\n--- Cancellation Rate by Market Segment ---")
print(cancel_by_market)

# Plot: Cancellation rate by market segment
plt.figure(figsize=(8, 4))
cancel_by_market.plot(kind="bar")
plt.title("Cancellation Percentage by Market Segment")
plt.xlabel("Market Segment")
plt.ylabel("Cancellations (%)")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# ----- Numerical Feature Correlations with Target -----
# Compute linear correlation of numerical features with the target variable
numeric_corr = df_hotel[numerical_cols].corr(numeric_only=True)["is_canceled"].sort_values(ascending=False)
print("\n--- Numerical Correlations with is_canceled ---")
print(numeric_corr)

# Plot: Feature correlation coefficients with the target variable
plt.figure(figsize=(8, 6))
numeric_corr.drop("is_canceled").plot(kind="barh")
plt.title("Numerical Features Correlation with is_canceled")
plt.xlabel("Correlation Coefficient")
plt.ylabel("Features")
plt.show()

## 🛠️ 4. Guidelines for Data Cleaning and Preprocessing

Basado en el diagnóstico y la auditoría de calidad de datos realizada previamente, se establecen las siguientes decisiones estratégicas para el diseño de nuestro pipeline:

### 💾 1. Integridad del Entorno y Arquitectura de Datos Crudos
* **Inmutabilidad del Origen:** El dataset original permanecerá completamente intacto y en modo de solo lectura dentro del directorio `data/raw/` para garantizar la reproducibilidad estructural en todas las iteraciones posteriores del experimento.

### 🛡️ 2. Mitigación de Sesgos y Defensa contra la Fuga de Datos (Data Leakage)
* **Remoción de Data Leakage:** Las características que actúan como sustitutos de la variable objetivo o que revelan información del futuro (`reservation_status`, `reservation_status_date`) se eliminan estrictamente antes de cualquier partición de datos. Esto asegura que el modelo solo evalúe variables conocidas en el momento real de la reserva.
* **Control de Redundancia (Preservación de Duplicados por Lógica de Negocio):** El dataset contiene 32,252 filas idénticas. Estos registros se conservan intencionalmente basándose en la lógica del dominio: en las operaciones hoteleras donde no existe un ID único de reserva, las filas idénticas representan reservas legítimas en bloque o al por mayor de operadores turísticos y eventos grupales. Eliminarlas subestimaría artificialmente el volumen y el alto impacto de cancelación del segmento de Grupos.
* **Aislamiento Estricto de Datos:** Para prevenir la contaminación de información, la partición de datos (Train-Test Split) se ejecuta *antes* de cualquier transformación de escala. Los parámetros de la distribución de entrenamiento se aíslan por completo del conjunto de prueba.

### ⚙️ 3. Reglas de la Arquitectura del Pipeline e Ingeniería de Características
Cada transformación de datos con estado se encapsula estrictamente de forma programática dentro del flujo de ejecución del pipeline:
* **Tratamiento de Valores Nulos (Estrategia de Imputación y Descarte):** * Los valores faltantes en características geográficas (ej. `country`) se resuelven inyectando una categoría controlada (`'Unknown'`) para preservar el registro.
  * El atributo operativo `agent` se imputa utilizando un identificador de respaldo (`0`), lo que significa reservas directas realizadas sin intermediarios.
  * La característica `company` se descarta por completo del pipeline debido a que presenta más del 94% de valores nulos, evitando la introducción de ruido estadístico o sesgos por imputación masiva.
* **Codificación Categórica (Expansión por One-Hot Encoding):** Los factores cualitativos categóricos se convierten mediante un One-Hot Encoding estricto. El motor de producción aplica una regla de alineación de esquema posterior (`.reindex`) para mantener una matriz determinista de 247 variables ordenadas alfabéticamente.
* **Escalamiento Estadístico (Geometría del StandardScaler):** Las variables numéricas se transforman mediante un `StandardScaler` centralizado. Esto alinea los valores a una estructura de media cero y varianza unitaria, optimizando la convergencia del descenso de gradiente para estimadores altamente sensibles a la magnitud, como la Regresión Logística y la Red Neuronal de TensorFlow.

### 4.1 Dataset Cleaning Execution

In [ ]:
# Anomaly detected
df_hotel = df_hotel[df_hotel['adr'] >= 0]
print(f"\n--> Data Cleaning: Anomalous negative ADR value removed. New dataset shape: {df_hotel.shape}")

In [ ]:

# Drop the 'company' column
df_hotel = df_hotel.drop(columns=['company'])

# Impute missing values in 'agent' with 0 
df_hotel['agent'] = df_hotel['agent'].fillna(0)

# Impute missing values in 'country' with 'Unknown'
df_hotel['country'] = df_hotel['country'].fillna('Unknown')

# Impute missing values in 'children' with 0
df_hotel['children'] = df_hotel['children'].fillna(0)

# Verify that the dataset is completely clean
print("Maximum number of remaining nulls in any column:")
print(df_hotel.isnull().sum().max())

## Data Preprocessing

### 4.2 Feature-target split (X and y)

In [ ]:
# Define 'X' (Independent Variables)
# We take the entire dataset but drop the target column
X = df_hotel.drop(columns=['is_canceled'])

# Define 'y' (Dependent Variable)
y = df_hotel['is_canceled']

# Verify the dimensions
print(f"Shape of X (Matrix): {X.shape}")
print(f"Shape of y (Vector): {y.shape}")

### 4.3 Train/Test Split

In [ ]:
# Split the dataset into Training (80%) and Testing (20%) sets using raw features
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Verify the dimensions of the generated splits
print("--- Training Set ---")
print(f"X_train shape (Features for learning): {X_train.shape}")
print(f"y_train shape (Labels for learning): {y_train.shape}\n")

print("--- Testing Set ---")
print(f"X_test shape (Features for the exam): {X_test.shape}")
print(f"y_test shape (Labels for grading): {y_test.shape}")

### 4.4 Categorical Encoding

In [ ]:
# (One-Hot Encoding)

# Automatically detect all columns containing text from the training split
text_columns = X_train.select_dtypes(include=['object', 'str']).columns
print("Categorical columns to be transformed:\n", list(text_columns))

In [ ]:
# Apply One-Hot Encoding independently to Training and Testing sets
X_train_encoded = pd.get_dummies(X_train, columns=text_columns, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, columns=text_columns, drop_first=True)

# Align columns to ensure both sets have the exact same features
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)

# Clean up column names by replacing spaces with underscores
X_train_encoded.columns = X_train_encoded.columns.str.replace(' ', '_')
X_test_encoded.columns = X_test_encoded.columns.str.replace(' ', '_')

In [ ]:
# Compare the shape of the datasets before and after the transformation
print(f"Dimensions of X_train before encoding: {X_train.shape}")
print(f"Dimensions of X_train after encoding:  {X_train_encoded.shape}")
print("-" * 50)
print(f"Dimensions of X_test before encoding:  {X_test.shape}")
print(f"Dimensions of X_test after encoding:   {X_test_encoded.shape}")

In [ ]:
# Print all column names
column_names = X_train_encoded.columns.tolist()
print("Total columns:", len(column_names))
print("List of columns:\n", column_names)

### 4.5 Feature Scaling

In [ ]:
# Automatically detect the original numerical columns from the source matrix X
num_columns = X.select_dtypes(include=['int64', 'float64']).columns
print("Numerical columns to be scaled:\n", list(num_columns))

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit and transform the Encoded Training matrix
X_train_encoded[num_columns] = scaler.fit_transform(X_train_encoded[num_columns])

# Transform the Encoded Testing matrix (using Train parameters to avoid leakage)
X_test_encoded[num_columns] = scaler.transform(X_test_encoded[num_columns])

In [ ]:
# Inspect the final output of the entire Data Preprocessing Phase
X_train_encoded.head()

In [ ]:
# Inspect the test matrix to verify successful scaling
X_test_encoded.head()

## 🧠 5. Model Training and Evaluation

### 5.1 Training the Baseline Models

#### Logistic Regression

In [ ]:
# =========================================================
# 1. LOGISTIC REGRESSION (Baseline Model 1)
# =========================================================
print("--- Training Logistic Regression Model ---")

# Initialize the model with a max_iter limit to ensure convergence
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# Train the model using the training data
log_reg.fit(X_train_encoded, y_train)

# Generate predictions on the unseen test set
y_pred_log_reg = log_reg.predict(X_test_encoded)

# Extract only column [:, 1], which represents the probability of cancellation (Class 1)
y_proba_log_reg = log_reg.predict_proba(X_test_encoded)[:, 1]

# Evaluate the performance metrics
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_log_reg):.4f}")
print(f"Logistic Regression ROC-AUC:  {roc_auc_score(y_test, y_proba_log_reg):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_log_reg))

print("\n" + "="*60 + "\n")

#### Decision Tree

In [ ]:
# =========================================================
# 2. DECISION TREE (Baseline Model 2)
# =========================================================
print("--- Training Decision Tree Model ---")

# Initialize the model with max_depth to control overfitting
tree_clf = DecisionTreeClassifier(max_depth=10, random_state=42)

# Train the model using the training data
tree_clf.fit(X_train_encoded, y_train)

# Generate predictions on the unseen test set
y_pred_tree = tree_clf.predict(X_test_encoded)

# Evaluate the performance metrics
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred_tree):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_tree))

print("\n" + "="*60 + "\n")

#### Random Forest

In [ ]:
# ==============================================================================
# 3. RANDOM FOREST (Advanced Ensemble Model 1)
# ==============================================================================
print("--- Training Random Forest Model ---")

# Initialize the ensemble model with tuned hyperparameters for performance
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)

# Train the model using the training data
rf_clf.fit(X_train_encoded, y_train)

# Generate predictions on the unseen test set
y_pred_rf = rf_clf.predict(X_test_encoded)

# Evaluate the performance metrics
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

print("\n" + "="*60 + "\n")

#### Training XGBoost

In [ ]:
# ==============================================================================
# 4. GRADIENT BOOSTING - XGBOOST (Advanced Ensemble Model 2)
# ==============================================================================
print("--- Training XGBoost Model ---")

# Initialize the extreme gradient boosting classifier
xgb_clf = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)

# Train the model using the training data
xgb_clf.fit(X_train_encoded, y_train)

# Generate predictions on the unseen test set
y_pred_xgb = xgb_clf.predict(X_test_encoded)

# Evaluate the performance metrics
print(f"XGBoost Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb))

print("\n" + "="*60 + "\n")

#### Deep Learning Model (Keras/TensorFlow)

In [ ]:
# ==============================================================================
# 5. MULTI-LAYER PERCEPTRON / NEURAL NETWORK (Deep Learning Model)
# ==============================================================================
print("--- Training Deep Learning Model (Keras/TensorFlow) ---")

# Define the network architecture
model = Sequential([
    # Input layer implicitly matching X_train shape, connected to the first hidden layer
    Dense(64, activation='relu', input_shape=(X_train_encoded.shape[1],)),
    Dropout(0.3), 
    
    # Second hidden layer
    Dense(32, activation='relu'),
    Dropout(0.2),
    
    Dense(1, activation='sigmoid')
])

# Compile the model 
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the neural network model
history = model.fit(
    X_train_encoded, y_train, 
    epochs=10, 
    batch_size=64, 
    validation_split=0.1, 
    verbose=1
)

# Generate probability predictions and convert them to binary classes (0 or 1)
y_pred_probs = model.predict(X_test_encoded)
y_pred_nn = (y_pred_probs > 0.5).astype(int).flatten()

# Evaluate the final performance metrics
print("\n" + "="*60 + "\n")
print(f"Neural Network Accuracy: {accuracy_score(y_test, y_pred_nn):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_nn))

### 5.2 Model Comparison Summary

In [ ]:
import pandas as pd
from sklearn.metrics import f1_score

# ==============================================================================
# FASE 4: MODEL COMPARISON AND FINAL SUMMARY
# ==============================================================================
print("--- Generating Model Comparison Summary ---")

# 1. Gather accuracy and F1-score metrics for all trained models
# Note: pos_label=1 extracts the specific performance for the cancellation class
results_data = {
    "Model": [
        "Logistic Regression", 
        "Decision Tree", 
        "Random Forest", 
        "XGBoost", 
        "Neural Network (Keras)"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_log_reg),
        accuracy_score(y_test, y_pred_tree),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_nn)
    ],
    "F1-Score (Canceled)": [
        f1_score(y_test, y_pred_log_reg, pos_label=1),
        f1_score(y_test, y_pred_tree, pos_label=1),
        f1_score(y_test, y_pred_rf, pos_label=1),
        f1_score(y_test, y_pred_xgb, pos_label=1),
        f1_score(y_test, y_pred_nn, pos_label=1)
    ]
}

# 2. Convert the dictionary into a formatted Pandas DataFrame
df_comparison = pd.DataFrame(results_data)

# 3. Sort the models by performance (highest Accuracy first)
df_comparison = df_comparison.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

# 4. Display the final comparison matrix
df_comparison

## 6. Technical Evaluation & Production Visualization

In [ ]:
# ==============================================================================
# SECTION: Technical Evaluation & Production Visualization
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve, auc

# Set beautiful plotting styles for production-grade reports
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 14})

# THE CONTROL SWITCH: Define who is the champion in the notebook
champion_model = xgb_clf  

# ------------------------------------------------------------------------------
# 1. Confusion Matrix with Operational Labels
# ------------------------------------------------------------------------------
y_pred_champion = champion_model.predict(X_test_encoded)
cm = confusion_matrix(y_test, y_pred_champion)

plt.figure(figsize=(7, 5.5))
sns.heatmap(
    cm, 
    annot=True, 
    fmt="d", 
    cmap="Blues", 
    cbar=False,
    xticklabels=["Not Canceled (0)", "Canceled (1)"],
    yticklabels=["Not Canceled (0)", "Canceled (1)"]
)
plt.title("Confusion Matrix with Operational Labels", pad=15, fontweight='bold')
plt.xlabel("Predicted Booking Status", labelpad=10)
plt.ylabel("Actual Booking Status", labelpad=10)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------------------------
# 2. Comparative ROC Curves
# ------------------------------------------------------------------------------
plt.figure(figsize=(8.5, 6.5))

# The dictionary remains intact because it draws ALL competitors to compare them
models_to_plot = {
    "Logistic Regression": log_reg,
    "Decision Tree": tree_clf,
    "Random Forest": rf_clf,
    "XGBoost": xgb_clf
}

for label, model_obj in models_to_plot.items():
    if hasattr(model_obj, "predict_proba"):
        y_proba = model_obj.predict_proba(X_test_encoded)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        roc_auc = auc(fpr, tpr)
        
        # Thicker line if this model matches the current champion_model object
        lw = 3.0 if model_obj == champion_model else 1.5
        label_text = f"{label} (Champion) (AUC = {roc_auc:.4f})" if model_obj == champion_model else f"{label} (AUC = {roc_auc:.4f})"
        
        plt.plot(fpr, tpr, label=label_text, linewidth=lw)

# Reference line representing absolute random guessing (baseline)
plt.plot([0, 1], [0, 1], color="darkgray", linestyle="--", label="Random Guessing (AUC = 0.5000)")

plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.title("Comparative ROC Curves", pad=15, fontweight='bold')
plt.xlabel("False Positive Rate (1 - Specificity)", labelpad=10)
plt.ylabel("True Positive Rate (Sensitivity)", labelpad=10)
plt.legend(loc="lower right", frameon=True)
plt.tight_layout()
plt.show()


# ------------------------------------------------------------------------------
# 3. Feature Importance Mapping (Champion Model)
# ------------------------------------------------------------------------------
# It automatically extracts the importances from whichever model is the champion
if hasattr(champion_model, "feature_importances_"):
    importances = champion_model.feature_importances_
    feature_names = X_train_encoded.columns
    indices = np.argsort(importances)[::-1][:15]

    plt.figure(figsize=(9, 6.5))
    sns.barplot(
        x=importances[indices], 
        y=feature_names[indices], 
        palette="viridis",
        hue=feature_names[indices],
        legend=False
    )
    plt.title("Feature Importance Mapping (Champion Model - Top 15)", pad=15, fontweight='bold')
    plt.xlabel("Relative Mathematical Importance Score", labelpad=10)
    plt.ylabel("Feature/Variable Name", labelpad=10)
    plt.tight_layout()
    plt.show()
else:
    print("The selected champion model architecture does not support feature_importances_ natively.")